# Atelier 2 — VIT — adaptation KARMAS Hammam

Version corrigée à réexécuter sur GPU. Les scores de référence ne valident pas cette version.

[Provenance du code et des résultats](../../references/eachr4f/PROVENANCE.md).

In [ ]:

from pathlib import Path
import os, sys, subprocess, importlib.util
ROOT = Path.cwd()
if not (ROOT / 'src/adaptation.py').exists():
    # Accepter un lancement depuis notebooks/colab/ ou depuis Colab lui-même.
    for candidate in list(ROOT.parents):
        if (candidate / 'src/adaptation.py').exists():
            ROOT = candidate
            break
    else:
        ROOT = Path('/content/Atelier2_DeepLearning') if Path('/content').exists() else Path('/kaggle/working/Atelier2_DeepLearning')
        if not ROOT.exists():
            subprocess.run(['git', 'clone', 'https://github.com/KarmasHammam/Atelier2_DeepLearning.git', str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = str(ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
if importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Active un GPU : Colab > Exécution > Modifier le type d’exécution > GPU.')
from src.adaptation import prepare_work, save_result
WORK_DIR = prepare_work('vit')
os.environ['ATELIER2_WORK_DIR'] = str(WORK_DIR)
print('Adaptation KARMAS Hammam ; source Eachr4f, commit 47654402f475fbe0e1d608b8e8417f28bb080d27')
print('Matériel de cette session :', torch.cuda.get_device_name(0))
print('Les résultats de référence importés ne sont pas ceux de cette nouvelle exécution.')


# Lab 2 — Partie 2 : Vision Transformer (ViT) sur MNIST

## Objectif

Construire et évaluer un Vision Transformer (ViT) pour classifier les chiffres de MNIST. Le code reprend le fichier `vit_torch.py` fourni et suit son architecture et ses hyperparamètres. Les cellules sont organisées pour présenter la démarche comme un rapport.

Les étapes pédagogiques s’appuient sur le [tutoriel ViT de Brian Pulfer](https://medium.com/@brianpulfer/vision-transformers-from-scratch-pytorch-a-step-by-step-guide-96c3313c2e0c).

**Kaggle :** sélectionnez un GPU dans *Settings → Accelerator*. Le notebook conserve le modèle et la boucle d’entraînement d’origine. Les seules adaptations d’exécution Kaggle sont le chemin des données (`/kaggle/working/datasets`) et la création des patches sur le même appareil que les images (CPU ou GPU).


## 1. Imports et reproductibilité
Importer PyTorch, torchvision et les utilitaires de progression. Les graines aléatoires sont fixées comme dans le fichier fourni.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets.mnist import MNIST
from torchvision.transforms import ToTensor
from tqdm import tqdm, trange

np.random.seed(42)
torch.manual_seed(42)


## 2. Découpage de l’image en patches
La fonction découpe chaque image en une grille de patches non chevauchants. Avec MNIST 28×28 et 7 patches par côté, chaque image devient une séquence de 49 patches de 4×4 pixels. Le découpage est vectorisé pour éviter les boucles par image et mieux exploiter le GPU.


In [ ]:
def patchify(images, n_patches):
    n, c, h, w = images.shape

    assert h == w, "Patchify method is implemented for square images only"
    assert h % n_patches == 0, "Image dimensions must be divisible by n_patches"

    patch_size = h // n_patches
    patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
    return patches.reshape(n, n_patches**2, -1)


## 3. Attention multi-têtes
`MyMSA` conserve les projections Q, K et V séparées par tête, mais calcule les sorties pour tout le mini-lot simultanément. Cette vectorisation conserve le calcul d’attention du modèle et réduit les boucles Python.


In [ ]:
class MyMSA(nn.Module):
    def __init__(self, d, n_heads=2):
        super(MyMSA, self).__init__()
        self.d = d
        self.n_heads = n_heads

        assert d % n_heads == 0, f"Can't divide dimension {d} into {n_heads} heads"

        d_head = int(d / n_heads)
        self.q_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.k_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.v_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.d_head = d_head
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequences):
        # sequences: (batch, sequence_length, token_dim)
        head_results = []
        for head in range(self.n_heads):
            start = head * self.d_head
            end = (head + 1) * self.d_head
            seq = sequences[:, :, start:end]
            q = self.q_mappings[head](seq)
            k = self.k_mappings[head](seq)
            v = self.v_mappings[head](seq)
            attention = self.softmax(q @ k.transpose(-2, -1) / (self.d_head ** 0.5))
            head_results.append(attention @ v)
        return torch.cat(head_results, dim=-1)


## 4. Bloc encodeur Transformer
Le bloc combine l’attention multi-têtes et un MLP, avec normalisation et connexions résiduelles.

In [ ]:
class MyViTBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMSA(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d),
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out


## 5. Modèle ViT
`MyViT` transforme les patches en tokens, ajoute le token de classification et les positions, applique les blocs Transformer, puis prédit l’une des dix classes.

In [ ]:
class MyViT(nn.Module):
    def __init__(self, chw, n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10):
        # Super constructor
        super(MyViT, self).__init__()

        # Attributes
        self.chw = chw  # ( C , H , W )
        self.n_patches = n_patches
        self.n_blocks = n_blocks
        self.n_heads = n_heads
        self.hidden_d = hidden_d

        # Input and patches sizes
        assert (
            chw[1] % n_patches == 0
        ), "Input shape not entirely divisible by number of patches"
        assert (
            chw[2] % n_patches == 0
        ), "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1) Linear mapper
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)

        # 2) Learnable classification token
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))

        # 3) Positional embedding
        self.register_buffer(
            "positional_embeddings",
            get_positional_embeddings(n_patches**2 + 1, hidden_d),
            persistent=False,
        )

        # 4) Transformer encoder blocks
        self.blocks = nn.ModuleList(
            [MyViTBlock(hidden_d, n_heads) for _ in range(n_blocks)]
        )

        # 5) Classification MLPk
        self.mlp = nn.Linear(self.hidden_d, out_d)

    def forward(self, images):
        # Dividing images into patches
        n, c, h, w = images.shape
        patches = patchify(images, self.n_patches).to(self.positional_embeddings.device)

        # Running linear layer tokenization
        # Map the vector corresponding to each patch to the hidden size dimension
        tokens = self.linear_mapper(patches)

        # Adding classification token to the tokens
        tokens = torch.cat((self.class_token.expand(n, 1, -1), tokens), dim=1)

        # Adding positional embedding
        out = tokens + self.positional_embeddings.repeat(n, 1, 1)

        # Transformer Blocks
        for block in self.blocks:
            out = block(out)

        # Getting the classification token only
        out = out[:, 0]

        return self.mlp(out)  # Logits attendus par CrossEntropyLoss


## 6. Encodage positionnel
L’encodage sinusoïdal fournit au Transformer la position de chaque patch dans la séquence. Cette fonction est appelée lorsque le modèle est créé dans la cellule suivante.

In [ ]:
def get_positional_embeddings(sequence_length, d):
    result = torch.ones(sequence_length, d)
    for i in range(sequence_length):
        for j in range(d):
            result[i][j] = (
                np.sin(i / (10000 ** (j / d)))
                if j % 2 == 0
                else np.cos(i / (10000 ** ((j - 1) / d)))
            )
    return result


## 7. Chargement des données
MNIST est chargé via `torchvision`. Le code télécharge les fichiers si nécessaire et les place dans le répertoire Kaggle accessible en écriture. Les ensembles officiels train et test sont utilisés.

In [ ]:
# Même partage 54 000 / 6 000 que les quatre autres modèles.
transform = ToTensor()
full_train_set = MNIST(root=str(WORK_DIR), train=True, download=False, transform=transform)
test_set = MNIST(root=str(WORK_DIR), train=False, download=False, transform=transform)
indices = torch.randperm(60000, generator=torch.Generator().manual_seed(42)).tolist()
train_set = torch.utils.data.Subset(full_train_set, indices[6000:])
val_set = torch.utils.data.Subset(full_train_set, indices[:6000])
train_loader = DataLoader(train_set, shuffle=True, batch_size=128, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, shuffle=False, batch_size=128, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, shuffle=False, batch_size=128, num_workers=2, pin_memory=True)
print(len(train_set), len(val_set), len(test_set))

## 8. Appareil, modèle et hyperparamètres
Le notebook détecte les GPU Kaggle. Si deux GPU sont disponibles, `DataParallel` répartit les mini-lots sur les GPU 0 et 1. Le modèle conserve les paramètres fournis : 7 patches par côté, 2 blocs, dimension cachée 8, 2 têtes, batch de 128, 5 époques et Adam à 0,005. Les patches et l’attention sont vectorisés pour réduire le travail Python.

La couche finale renvoie des logits, attendus par `CrossEntropyLoss`. Cette correction ne change pas l’architecture du ViT.


In [ ]:
# Configuration du matériel et du modèle
num_gpus = torch.cuda.device_count()
device = torch.device("cuda:0" if num_gpus > 0 else "cpu")
print("Using device:", device)
print("Nombre de GPU détectés:", num_gpus)
for gpu_id in range(num_gpus):
    print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")

model = MyViT(
    (1, 28, 28), n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10
).to(device)
if num_gpus >= 2:
    model = nn.DataParallel(model, device_ids=[0, 1])
    print("DataParallel activé sur GPU 0 et GPU 1.")
elif num_gpus == 1:
    print("Un seul GPU est disponible : entraînement sur GPU 0.")
else:
    print("Aucun GPU détecté : entraînement sur CPU.")

N_EPOCHS = 5
LR = 0.005


## 9. Entraînement et suivi de la validation

Le réseau apprend sur les 54 000 images du train. La loss de validation est
calculée après chaque époque sur 6 000 autres images. Les meilleurs poids sont
sélectionnés selon cette loss, puis restaurés avant l'évaluation finale du test.

In [ ]:
import time
# Entraînement : mêmes hyperparamètres que le fichier fourni.
optimizer = Adam(model.parameters(), lr=LR)
criterion = CrossEntropyLoss()
train_losses = []
val_losses = []
best_val_loss = float("inf")
best_state = None
torch.cuda.synchronize()
training_start_time = time.time()

def evaluate_loss(loader):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            total_loss += criterion(logits, y).item() * y.size(0)
            total_samples += y.size(0)
    return total_loss / total_samples

for epoch in trange(N_EPOCHS, desc="Training"):
    model.train()
    train_loss_sum = 0.0
    train_samples = 0

    for x, y in tqdm(
        train_loader, desc=f"Epoch {epoch + 1} in training", leave=False
    ):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.detach().item() * y.size(0)
        train_samples += y.size(0)

    train_epoch_loss = train_loss_sum / train_samples
    val_epoch_loss = evaluate_loss(val_loader)
    train_losses.append(train_epoch_loss)
    val_losses.append(val_epoch_loss)
    if val_epoch_loss < best_val_loss:
        best_val_loss = val_epoch_loss
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    print(
        f"Epoch {epoch + 1}/{N_EPOCHS} | "
        f"train loss: {train_epoch_loss:.4f} | "
        f"val loss: {val_epoch_loss:.4f}"
    )

torch.cuda.synchronize()
training_seconds = time.time() - training_start_time
print(f"Temps total d’entraînement : {training_seconds / 60:.2f} minutes ({training_seconds:.1f} secondes)")

# Restaurer le checkpoint choisi sur validation, jamais sur le test.
model.load_state_dict(best_state)


## 10. Courbes train et validation

Le temps mesure entraînement, validation et sélection des poids. Le test ne
participe pas à ces courbes ; il reste réservé à l'évaluation finale.

In [ ]:
epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker="o", label="Train loss")
plt.plot(epochs, val_losses, marker="o", label="Validation loss")
plt.title("Évolution de la loss — ViT MNIST")
plt.xlabel("Époque")
plt.ylabel("Cross Entropy Loss")
plt.xticks(list(epochs))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 11. Évaluation finale sur le test
Après l’entraînement, cette cellule calcule la loss, l’accuracy et le F1 macro sur les 10 000 images test.


In [ ]:
# Évaluation finale : loss, accuracy et F1 macro
from sklearn.metrics import f1_score

model.eval()
with torch.no_grad():
    correct, total = 0, 0
    test_loss = 0.0
    all_targets, all_predictions = [], []
    for batch in tqdm(test_loader, desc="Testing"):
        x, y = batch
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        y_hat = model(x)
        loss = criterion(y_hat, y)
        predictions = torch.argmax(y_hat, dim=1)
        test_loss += loss.detach().cpu().item() * len(y)

        correct += torch.sum(predictions == y).detach().cpu().item()
        total += len(x)
        all_targets.extend(y.detach().cpu().tolist())
        all_predictions.extend(predictions.detach().cpu().tolist())

    test_loss /= total
    test_f1_macro = f1_score(all_targets, all_predictions, average="macro")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {correct / total * 100:.2f}%")
    print(f"F1 macro: {test_f1_macro:.4f}")


## 12. Prédictions sur quatre images test
Chaque vignette montre l’image, sa classe réelle et la classe prédite. Le vert indique une prédiction correcte et le rouge une erreur.


In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    predictions = model(images[:4].to(device, non_blocking=True)).argmax(dim=1).cpu()

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, image, label, prediction in zip(axes, images[:4], labels[:4], predictions):
    true_label = label.item()
    predicted_label = prediction.item()
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(
        f"Réel : {true_label}\nPrédit : {predicted_label}",
        color="green" if true_label == predicted_label else "red"
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## Interprétation après cette nouvelle exécution

Comparer la diminution de la loss train et celle de validation, puis analyser
l'accuracy et le F1 du test. Une divergence train/validation peut signaler du
surapprentissage. Les limites du modèle compact doivent être discutées sans
conclure à la supériorité générale d'une famille de modèles.

Les 87,92 % d'accuracy et le F1 de 0,8772 appartiennent à l'exécution d'origine,
conservée dans references/eachr4f/. Cette version utilise un autre partage et une
sélection sur validation : ses propres résultats seront ceux affichés après
exécution, et non ces valeurs de référence.

## Enregistrer cette nouvelle exécution

Les métriques sont calculées depuis les prédictions de cette session GPU. Le dossier
`results/adapted_gpu/` reste distinct des résultats importés et des anciens profils.
Enregistrer aussi le notebook dans GitHub **avec ses sorties**. L'archive permet de
conserver les résultats, même si les cinq modèles ne sont pas tous terminés.

In [ ]:
result = save_result('vit', all_targets, all_predictions, test_loss, training_seconds, WORK_DIR)
from scripts.export_adapted_results import export
archive = export(allow_partial=True)
print('Archive :', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Archive disponible localement.')